[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/practice/intermediate/02_model_selection_and_evaluation.ipynb)

# Luyện tập trung cấp 2 — Chọn mô hình và đánh giá đúng cách

Mỗi bài là một **tình huống**: chọn sơ đồ chia dữ liệu, metric, cách tinh chỉnh phù hợp — rồi cài đặt. Sai ở bước này làm mọi con số phía sau vô nghĩa.

Chương liên quan: 14, 15, 16, 19, 37.

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/practice/intermediate

In [ ]:
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import (GridSearchCV, GroupKFold, KFold, RandomizedSearchCV, StratifiedKFold, TimeSeriesSplit,
                                     cross_val_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true

rng = np.random.default_rng(0)

**Bài 1 — Dữ liệu có nhóm.** Ảnh y tế: 60 bệnh nhân × 10 ảnh; ảnh của cùng bệnh nhân rất giống nhau (đặc trưng "riêng" của bệnh nhân). So sánh AUC 5-fold của random forest khi chia **theo ảnh** (`KFold(shuffle=True, random_state=0)`) với chia **theo bệnh nhân** (`GroupKFold(5)`) → `auc_by_image`, `auc_by_patient`. Con số nào đáng tin?

In [ ]:
patients = np.repeat(np.arange(60), 10)
patient_signature = rng.normal(size=(60, 5))[patients]               # mỗi bệnh nhân một "dấu vân tay"
label_by_patient = rng.integers(0, 2, 60)
X1 = patient_signature + rng.normal(0, 0.3, (600, 5))
X1[:, 0] += 0.5 * label_by_patient[patients]                           # tín hiệu bệnh yếu
y1 = label_by_patient[patients]
forest = RandomForestClassifier(200, random_state=0, n_jobs=-1)
### START CODE HERE ### (≈ 2 dòng)
auc_by_image = cross_val_score(forest, X1, y1, cv=KFold(5, shuffle=True, random_state=0), scoring="roc_auc").mean()
auc_by_patient = cross_val_score(forest, X1, y1, cv=GroupKFold(5), groups=patients, scoring="roc_auc").mean()
### END CODE HERE ###
print(f"chia theo ảnh: {auc_by_image:.3f} | chia theo bệnh nhân: {auc_by_patient:.3f}")

In [ ]:
check_greater(auc_by_image, 0.9, "chia theo ảnh: mô hình nhận ra bệnh nhân → điểm ảo")
check_less(auc_by_patient, auc_by_image - 0.2, "chia theo bệnh nhân: đánh giá trung thực, thấp hơn nhiều")

**Bài 2 — Dữ liệu chuỗi thời gian.** Dự đoán giá trị ngày mai từ 7 lag. So MAE của Ridge với `KFold(shuffle=True)` và `TimeSeriesSplit(5)` → `mae_shuffled`, `mae_time`. (Chuỗi có xu hướng và đổi chế độ giữa chừng.)

In [ ]:
t = np.arange(1200)
series = 0.02 * t + np.where(t > 800, 8 * np.sin(t / 15), 2 * np.sin(t / 15)) + rng.normal(0, 1, 1200)
lags = np.column_stack([np.roll(series, k) for k in range(1, 8)])[7:]
target = series[7:]
ridge = make_pipeline(StandardScaler(), Ridge(1.0))
### START CODE HERE ### (≈ 2 dòng)
mae_shuffled = -cross_val_score(ridge, lags, target, cv=KFold(5, shuffle=True, random_state=0), scoring="neg_mean_absolute_error").mean()
mae_time = -cross_val_score(ridge, lags, target, cv=TimeSeriesSplit(5), scoring="neg_mean_absolute_error").mean()
### END CODE HERE ###
print(f"MAE xáo trộn: {mae_shuffled:.3f} | theo thời gian: {mae_time:.3f}")

In [ ]:
check_greater(mae_time, mae_shuffled, "đánh giá theo thời gian khắt khe hơn (thế giới đổi chế độ)")

Xáo trộn cho mô hình học từ **tương lai** (cả giai đoạn sau khi đổi chế độ) rồi đánh giá trên quá khứ xen kẽ — lạc quan. TimeSeriesSplit mô phỏng đúng việc dự báo: chỉ học từ quá khứ.

**Bài 3 — Tinh chỉnh với ngân sách.** Random forest trên dữ liệu `X3`; so sánh `GridSearchCV` (lưới 3×3 trên `max_depth`, `min_samples_leaf`) và `RandomizedSearchCV` (9 lần thử, phân phối rộng hơn, `random_state=0`) — cùng **9 lần thử**, 3-fold, ROC-AUC → `grid_best`, `random_best` (best_score_).

In [ ]:
X3 = rng.normal(size=(800, 10))
y3 = ((X3[:, 0] * X3[:, 1] + X3[:, 2] ** 2 + 0.5 * rng.normal(size=800)) > 1).astype(int)
base = RandomForestClassifier(100, random_state=0, n_jobs=-1)
### START CODE HERE ### (≈ 6 dòng)
grid = GridSearchCV(base, {"max_depth": [2, 3, 4], "min_samples_leaf": [20, 50, 100]}, cv=3, scoring="roc_auc").fit(X3, y3)
grid_best = grid.best_score_
search = RandomizedSearchCV(base, {"max_depth": [2, 3, 4, 6, 8, 12, None], "min_samples_leaf": [1, 2, 5, 10, 20, 50, 100]},
                            n_iter=9, cv=3, scoring="roc_auc", random_state=0).fit(X3, y3)
random_best = search.best_score_
### END CODE HERE ###
print(f"grid: {grid_best:.3f} ({grid.best_params_}) | random: {random_best:.3f} ({search.best_params_})")

In [ ]:
check_equal(len(grid.cv_results_["params"]), 9, "lưới 9 cấu hình")
check_greater(random_best, grid_best, "lưới đặt sai vùng; random khám phá rộng hơn với cùng ngân sách")

**Bài 4 — Lời nguyền người thắng và nested CV.** Dữ liệu 120 mẫu, 100 đặc trưng **nhiễu thuần** (không có tín hiệu: AUC thật = 0.5). Một nhóm thử 40 tập 5 đặc trưng ngẫu nhiên cho hồi quy logistic, chọn tập có AUC 3-fold CV cao nhất và báo cáo chính điểm đó → `naive_auc`. Hãy ước lượng trung thực hiệu năng **của cả quy trình chọn** bằng nested CV: vòng ngoài `StratifiedKFold(5, shuffle=True, random_state=1)`; trong mỗi vòng, chọn tập tốt nhất bằng `cv_auc` **chỉ trên phần train**, huấn luyện trên phần train, chấm AUC trên phần test → `nested_auc` (trung bình 5 vòng).

In [ ]:
r4 = np.random.default_rng(4)
X4, y4 = r4.normal(size=(120, 100)), r4.integers(0, 2, 120)
subsets = [r4.choice(100, 5, replace=False) for _ in range(40)]


def cv_auc(X, y, cols):
    return cross_val_score(LogisticRegression(max_iter=1000), X[:, cols], y, cv=StratifiedKFold(3, shuffle=True, random_state=0), scoring="roc_auc").mean()


naive_auc = max(cv_auc(X4, y4, s) for s in subsets)
### START CODE HERE ### (≈ 6 dòng)
# STARTER: nested_auc = None
outer_scores = []
for tr, te in StratifiedKFold(5, shuffle=True, random_state=1).split(X4, y4):
    best = max(subsets, key=lambda s: cv_auc(X4[tr], y4[tr], s))
    model = LogisticRegression(max_iter=1000).fit(X4[tr][:, best], y4[tr])
    outer_scores.append(roc_auc_score(y4[te], model.predict_proba(X4[te][:, best])[:, 1]))
nested_auc = float(np.mean(outer_scores))
### END CODE HERE ###
print(f"điểm CV của tập 'tốt nhất' (lạc quan): {naive_auc:.3f} | nested CV: {nested_auc:.3f} (sự thật: 0.5)")

In [ ]:
check_greater(naive_auc, 0.55, "chọn tốt nhất trong 40 → điểm CV trông như có tín hiệu")
check_greater(naive_auc, nested_auc + 0.05, "nested CV lộ ra độ lạc quan")
check_less(abs(nested_auc - 0.5), 0.12, "nested CV gần sự thật (không có tín hiệu)")

Chọn **cái tốt nhất** trong nhiều ứng viên theo cùng một điểm CV làm điểm đó bị chệch lên (một phần là may) — dù dữ liệu không có tín hiệu nào. Muốn biết quy trình "thử nhiều cấu hình rồi chọn" thực sự tốt đến đâu, phải đưa **việc chọn** vào bên trong vòng đánh giá (nested CV) hoặc giữ một tập test chưa từng dùng để chọn (chương 15, 39).

**Bài 5 — Metric cho dữ liệu lệch.** Mô hình A và B trên dữ liệu 2% dương: tính ROC-AUC và PR-AUC của mỗi mô hình → `metrics` (DataFrame index A/B, cột `roc_auc`, `pr_auc`). Mô hình nào tốt hơn cho việc "đưa top 1% cảnh báo cho người kiểm tra"?

In [ ]:
n = 20000
y5 = (rng.random(n) < 0.02).astype(int)
score_a = y5 * rng.normal(2.0, 1, n) + (1 - y5) * rng.normal(0, 1, n)                 # tách đều
score_b = np.where(y5 == 1, np.where(rng.random(n) < 0.5, rng.normal(6, 0.5, n), rng.normal(0.3, 1, n)), rng.normal(0, 1, n))  # một nửa ca dương rất rõ
### START CODE HERE ### (≈ 3 dòng)
metrics = pd.DataFrame({"roc_auc": [roc_auc_score(y5, score_a), roc_auc_score(y5, score_b)],
                        "pr_auc": [average_precision_score(y5, score_a), average_precision_score(y5, score_b)]}, index=["A", "B"])
### END CODE HERE ###
top = lambda s: y5[np.argsort(-s)[: n // 100]].mean()  # noqa: E731
print(metrics.round(3), f"\nprecision trong top 1%: A {top(score_a):.2f} | B {top(score_b):.2f}")

In [ ]:
check_greater(metrics.loc["A", "roc_auc"], metrics.loc["B", "roc_auc"], "ROC-AUC chọn A")
check_greater(metrics.loc["B", "pr_auc"], metrics.loc["A", "pr_auc"], "PR-AUC chọn B — đúng với nhu cầu top cảnh báo")

**Bài 6 — Calibration.** Mỗi tín hiệu thật được đo bằng **ba cảm biến** gần trùng nhau (đặc trưng tương quan mạnh). So sánh Gaussian Naive Bayes với hồi quy logistic: dùng `calibration_curve` (10 bin, `strategy="quantile"`) trên test, tính sai số calibration trung bình tuyệt đối `cal_error` (dict tên → mean|tỉ lệ thật − xác suất TB|).

In [ ]:
r6 = np.random.default_rng(6)
signal = r6.normal(size=(14000, 2))
X6 = np.column_stack([signal[:, [0]] + 0.3 * r6.normal(size=(14000, 3)), signal[:, [1]] + 0.3 * r6.normal(size=(14000, 3))])
y6 = (r6.random(14000) < 1 / (1 + np.exp(-(1.2 * signal[:, 0] - 0.8 * signal[:, 1])))).astype(int)
X6_tr, X6_te, y6_tr, y6_te = X6[:4000], X6[4000:], y6[:4000], y6[4000:]
models6 = {"naive_bayes": GaussianNB(), "logistic": LogisticRegression()}
### START CODE HERE ### (≈ 4 dòng)
cal_error = {}
for name, m in models6.items():
    frac, mean_pred = calibration_curve(y6_te, m.fit(X6_tr, y6_tr).predict_proba(X6_te)[:, 1], n_bins=10, strategy="quantile")
    cal_error[name] = float(np.mean(np.abs(frac - mean_pred)))
### END CODE HERE ###
print({k: round(v, 3) for k, v in cal_error.items()})

In [ ]:
check_less(cal_error["logistic"], 0.03, "logistic hiệu chỉnh tốt")
check_greater(cal_error["naive_bayes"], 3 * cal_error["logistic"], "Naive Bayes quá tự tin khi đặc trưng tương quan")

Naive Bayes giả định các đặc trưng **độc lập** khi biết lớp; ba cảm biến đo cùng một tín hiệu bị coi là ba bằng chứng độc lập → bằng chứng bị **đếm ba lần**, xác suất bị đẩy về 0/1 (quá tự tin) dù thứ hạng vẫn tốt. Nếu cần xác suất (ngưỡng theo chi phí, kết hợp với hệ thống khác), hãy kiểm tra calibration và hiệu chỉnh lại (Platt/isotonic — chương 37).